<a href="https://colab.research.google.com/github/nurdalilahanan/Data-Management-Project-1/blob/main/NurDalilaHanan_iris_spark_mllib_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Iris Classification Using Spark MLlib**

### Project Overview



---


The objective of this project is to classify Iris flower species by utilising Apache Spark MLlib to implement a supervised classification workflow. The primary goal is to illustrate the application of scalable machine learning techniques in a distributed computational environment even for structured datset that are relatively small. The workflow incorporates the entire modelling pipeline, which includes feature transformation, model development, and optimisation. In order to facilitate a meaningful comparison between linear, rule-based, and ensemble learning paradigms, three classification algorithms which are Logistic Regression, Decision Tree, and Random Forest are chosen to represent distinct modelling approaches.

In contrast to the mere implementation of a model, this initiative prioritises methodological consistency. The evaluation of each model under optimised conditions is ensured through the systematic use of grid search and cross-validation for hyperparameter tuning. A comprehensive evaluation of model performance beyond a single metric perspective is achieved by employing multiple evaluation metrics, such as precision, recall, accuracy, and F1-score. The project's objective is to critically evaluate the trade offs between model complexity, interpretability, and generalisability, reflecting practical considerations in real-world data science applications, in addition to identifying the best-performing model, through this structured approach.

### Dataset Description


---



This study utilises the Iris dataset originally provided by Ronald A.Fisher. The dataset has 150 observations of three Iris flower species which are Setosa, Versicolour and Virginica with each class evenly represented. Each observation is characterised by four continuous numerical attributes which are sepal length, sepal width, petal length and petal width. These characteristics capture significant botanical traits that facilitate species differentiation. The dataset's enduring relevance is derived from its capacity to establish a clear and interpretable foundation for the evaluation of classification models despite its relatively small size and well structured nature. Consequently, it is a widely acknowledged benchmark in the field of machine learning.

The Iris dataset possesses a number of properties that are particularly advantageous for classification tasks from a data perspective. It maintains a balanced class distribution and does not contain any absent values which minimises the necessity for extensive preprocessing or resampling techniques. Simulataneously, the dataset exhibits linearly separable patterns particularly for the Setosa class as well as areas of overlap between Versicolour and Virginica, thereby introducing a moderate level of classification complexity. This combination enables various machine learning models to exhibit their respective strengths and limitations. The dataset functions as both a predictive task and a controlled experimental environment in this project, allowing for the comparison of the responses of various algorithms to structured, low dimensional data within the Apache Spark MLlib framework.

In [45]:
from google.colab import files
uploaded = files.upload()

Saving iris.data to iris (1).data


### Spark Session Setup

In [46]:
!pip install pyspark

In [47]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Iris Classification using Spark MLlib") \
    .getOrCreate()

spark

### Data Loading

/write/

In [48]:
# Import required Library
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [49]:
# Load Dataset into Spark
irisdf = spark.read.csv(
    "iris.data",
    inferSchema=True
)

irisdf.show(5)

+---+---+---+---+-----------+
|_c0|_c1|_c2|_c3|        _c4|
+---+---+---+---+-----------+
|5.1|3.5|1.4|0.2|Iris-setosa|
|4.9|3.0|1.4|0.2|Iris-setosa|
|4.7|3.2|1.3|0.2|Iris-setosa|
|4.6|3.1|1.5|0.2|Iris-setosa|
|5.0|3.6|1.4|0.2|Iris-setosa|
+---+---+---+---+-----------+
only showing top 5 rows


### Data Preprocessing


---



In [50]:
irisdf = irisdf.toDF(
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width",
    "species"
)

irisdf.show(5)

+------------+-----------+------------+-----------+-----------+
|sepal_length|sepal_width|petal_length|petal_width|    species|
+------------+-----------+------------+-----------+-----------+
|         5.1|        3.5|         1.4|        0.2|Iris-setosa|
|         4.9|        3.0|         1.4|        0.2|Iris-setosa|
|         4.7|        3.2|         1.3|        0.2|Iris-setosa|
|         4.6|        3.1|         1.5|        0.2|Iris-setosa|
|         5.0|        3.6|         1.4|        0.2|Iris-setosa|
+------------+-----------+------------+-----------+-----------+
only showing top 5 rows


In [51]:
# Remove Empty Row
from pyspark.sql.functions import col
irisdf = irisdf.filter(col("species").isNotNull())

In [52]:
# Check Data
irisdf.printSchema()

irisdf.describe().show()

irisdf.groupBy("species").count().show()

root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- species: string (nullable = true)

+-------+------------------+-------------------+------------------+------------------+--------------+
|summary|      sepal_length|        sepal_width|      petal_length|       petal_width|       species|
+-------+------------------+-------------------+------------------+------------------+--------------+
|  count|               150|                150|               150|               150|           150|
|   mean| 5.843333333333335| 3.0540000000000007|3.7586666666666693|1.1986666666666672|          NULL|
| stddev|0.8280661279778637|0.43359431136217375| 1.764420419952262|0.7631607417008414|          NULL|
|    min|               4.3|                2.0|               1.0|               0.1|   Iris-setosa|
|    max|               7.9|                4.4|               6.9|

In [53]:
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler

label_indexer = StringIndexer(inputCol="species", outputCol="label")

assembler = VectorAssembler(
    inputCols=["sepal_length", "sepal_width", "petal_length", "petal_width"],
    outputCol="raw_features"
)

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features"
)

The original Iris data file was initially loaded into a Spark DataFrame during data preprocessing, and meaningful header names were manually designated by hand, as the source file lacked column labels. The columns were redesignated as sepal_length, sepal_width, petal_length, petal_width, and species to facilitate interpretation and modelling. Finally, a data validation phase was implemented to verify the schema, missing values, and potential empty rows. According to the schema, the target variable, species, was recorded as a categorical string, while all four predictor variables were correctly identified as numerical double data types. The descriptive statistics indicated that all feature columns contained 150 complete records, and the missing value check did not yield any missing values. The class distribution was also balanced, with 50 observations for each Iris species: Iris-setosa, Iris-versicolor, and Iris-virginica. This suggests that the dataset was clean and did not necessitate imputation, duplicate treatment, or class balancing prior to model development.

Following the validation of the data quality, feature preparation was implemented to convert the dataset into the format necessary for Spark MLlib execution. The categorical species column was converted into a numerical label column using StringIndexer, as Spark classification models necessitate a numeric target variable. The four numerical input variables were subsequently combined into a single feature vector named raw_features using VectorAssembler, as Spark MLlib anticipates that predictors be stored in vector form. StandardScaler was subsequently incorporated to standardise the feature values, particularly to support models like Logistic Regression that may be sensitive to disparities in feature scale. The application of a consistent preprocessing pipeline guarantees a fair and reproducible modelling methodology across all selected algorithms despite the fact that tree-based models such as Decision Tree and Random Forest do not strictly require feature scaling. In general, the preprocessing stage guaranteed that the dataset was technically prepared for classification using Spark MLlib was clean and was accurately structured.

### Train-Test Split


---
A reliable evaluation of model performance on unseen data was facilitated by partitioning the dataset into training and testing subsets in an 80:20 ratio. This arrangement guarantees that the majority of the data is utilised to acquire knowledge of the fundamental patterns, while a distinct hold-out set is designated for the evaluation of generalisation capability. In order to guarantee reproducibility of results which is especially critical in experimental operations, a fixed random seed was implemented during the splitting process. It is anticipated that the random division will maintain proportional representation across all three Iris species in both subsets given the balanced class distribution of the dataset. This method facilitates a fair assessment of the models by mitigating sampling bias and offering a consistent foundation for comparing the predictive capabilities of various algorithms.


In [123]:
traindf, testdf = irisdf.randomSplit([0.8, 0.2], seed=42)

print("Training data:", traindf.count())
print("Testing data:", testdf.count())

Training data: 126
Testing data: 24


### Model Development and Tuning

#### Model 1 : Logistic Regression


---

Logistic Regression was chosen as the baseline linear classifier because of its interpretability and efficacy in modelling relationships where class boundaries are approximately linear. The model was trained within the Spark MLlib pipeline by transforming the categorical target variable into numerical form using StringIndexer, assembling the feature columns via VectorAssembler, and applying StandardScaler to ensure that optimisation processes are not influenced by differences in feature scale. Grid search was employed to tune hyperparameters including regularisation strength and elastic net blending within a cross-validation framework. The selection of numFolds = 5 is indicative of a compromise between the robustness of evaluation and the computational efficiency of the dataset. This approach enables each observation to contribute to both the training and validation phases thereby reducing the variance in performance estimates despite the relatively small size of the dataset. This ensures that the selected model parameters generalise well beyond a single train-test split.

In [124]:
# Define Model

from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label")

In [125]:
# Create Pipeline

from pyspark.ml import Pipeline
pipeline_lr = Pipeline(stages=[label_indexer, assembler, scaler, lr])

In [126]:
# Hyperparameter Tuning
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
paramGrid_lr = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.0, 0.01, 0.1]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

In [127]:
# Evaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
evaluator = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")

In [128]:
# Cross Validation
cv_lr = CrossValidator(
    estimator=pipeline_lr,
    estimatorParamMaps=paramGrid_lr,
    evaluator=evaluator,
    numFolds=5
)

modellr = cv_lr.fit(traindf)

In [129]:
# Predictions
predlr = modellr.transform(testdf)

In [130]:
# Evaluation
accuracylr = evaluator.evaluate(predlr)
print("Logistic Regression Accuracy:", accuracylr)

Logistic Regression Accuracy: 1.0


#### Model 2 : Decision Tree


---
The Decision Tree model was implemented to account for non-linear relationships and interaction effects that may not be adequately captured by linear models. It is appropriate for comprehending the role of feature thresholds in classification decisions due to its rule-based structure which provides intuitive interpretability. The modelling process adhered to a consistent preprocessing pipeline with hyperparameters such as maximum depth being optimised through grid search. Decision Trees are particularly liable to overfitting particularly on smaller datasets which is why the use of numFolds = 5 in cross-validation is crucial. The approach enhances the stability and generalisability of the final model by evaluating model performance across multiple folds thereby mitigating the risk of selecting a tree structure that performs well only on a specific subset of the data.


In [131]:
# Define Model
from pyspark.ml.classification import DecisionTreeClassifier
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label")

In [132]:
# Create Pipeline
pipeline_dt = Pipeline(stages=[label_indexer, assembler, scaler, dt])

In [133]:
# Hyperparameter Tuning
paramGrid_dt = ParamGridBuilder() \
    .addGrid(dt.maxDepth, [2, 3, 5, 7]) \
    .build()

In [134]:
# Cross Validation
cv_dt = CrossValidator(
    estimator=pipeline_dt,
    estimatorParamMaps=paramGrid_dt,
    evaluator=evaluator,
    numFolds=5
)

modeldt = cv_dt.fit(traindf)

In [135]:
# Predictions
preddt = modeldt.transform(testdf)

In [136]:
# Evaluation
accuracydt = evaluator.evaluate(preddt)
print("Decision Tree Accuracy:", accuracydt)

Decision Tree Accuracy: 1.0


#### Model 3 : Random Forest


---
Random Forest was selected as an ensemble extension of Decision Trees to enhance predictive performance and decrease variance by aggregating multiple tree models. Random Forest typically accomplishes superior generalisation in comparison to a single Decision Tree by integrating predictions from multiple trees that have been trained on distinct subsets of the data and feature space. The implementation maintained consistency by employing the same preprocessing pipeline which was subsequently optimised by hyperparameter tuning of critical parameters including the number of trees and tree depth. In order to accurately estimate the performance of this more intricate model while managing computational costs, the adoption of numFolds = 5 in cross-validation is justified. Random Forest introduces additional randomness through bootstrapping and feature sampling. Therefore, evaluating it across numerous folds stabilises performance estimates and guarantees that the chosen configuration is not excessively reliant on a specific data partition.


In [137]:
# Define Model
from pyspark.ml.classification import RandomForestClassifier
rf = RandomForestClassifier(featuresCol="features", labelCol="label")

In [138]:
# Create Pipeline
pipelinerf = Pipeline(stages=[label_indexer, assembler, scaler, rf])

In [139]:
# Hyperparameter Tuning
paramGrid_rf = ParamGridBuilder() \
    .addGrid(rf.numTrees, [10, 50, 100]) \
    .addGrid(rf.maxDepth, [3, 5, 7]) \
    .build()

In [140]:
# Cross Validation
cv_rf = CrossValidator(
    estimator=pipelinerf,
    estimatorParamMaps=paramGrid_rf,
    evaluator=evaluator,
    numFolds=5
)

modelrf = cv_rf.fit(traindf)

In [141]:
# Predictions
predrf = modelrf.transform(testdf)

In [142]:
# Evaluation
accuracyrf = evaluator.evaluate(predrf)
print("Random Forest Accuracy:", accuracyrf)

Random Forest Accuracy: 0.9583333333333334


### Model Evaluation

In [143]:
metrics = ["accuracy", "weightedPrecision", "weightedRecall", "f1"]

In [144]:
def evaluate_model(predictions, model_name):
    results = {}
    for m in metrics:
        evaluator.setMetricName(m)
        results[m] = evaluator.evaluate(predictions)
    print(f"\n{model_name} Performance:")
    for k, v in results.items():
        print(f"{k}: {v:.4f}")
    return results

In [145]:
resultslr = evaluate_model(predlr, "Logistic Regression")


Logistic Regression Performance:
accuracy: 1.0000
weightedPrecision: 1.0000
weightedRecall: 1.0000
f1: 1.0000


In [146]:
resultsdt = evaluate_model(preddt, "Decision Tree")


Decision Tree Performance:
accuracy: 1.0000
weightedPrecision: 1.0000
weightedRecall: 1.0000
f1: 1.0000


In [147]:
resultsrf = evaluate_model(predrf, "Random Forest")


Random Forest Performance:
accuracy: 0.9583
weightedPrecision: 0.9635
weightedRecall: 0.9583
f1: 0.9578


### Comparative Analysis

In [148]:
import pandas as pd

In [149]:
comparisondf = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest"],
    "Accuracy": [resultslr["accuracy"], resultsdt["accuracy"], resultsrf["accuracy"]],
    "Precision": [resultslr["weightedPrecision"], resultsdt["weightedPrecision"], resultsrf["weightedPrecision"]],
    "Recall": [resultslr["weightedRecall"], resultsdt["weightedRecall"], resultsrf["weightedRecall"]],
    "F1-Score": [resultslr["f1"], resultsdt["f1"], resultsrf["f1"]]
})

In [150]:
comparisondf

,Model,Accuracy,Precision,Recall,F1-Score
0,Logistic Regression,1.000000,1.000000,1.000000,1.000000
1,Decision Tree,1.000000,1.000000,1.000000,1.000000
2,Random Forest,0.958333,0.963542,0.958333,0.957828


#### *Interpretation*

The accuracy, precision, recall, and F1-score of the three classification models which are Logistic Regression, Decision Tree, and Random Forest were assessed using a variety of metrics. The results of the evaluation indicate that both Logistic Regression and Decision Tree achieved perfect scores across all evaluation metrics signifying flawless classification on the testing dataset. This implies that the Iris dataset's fundamental structure is adequately well-defined to enable both linear and non-linear models to effectively capture the decision boundaries. The partial linear separability of the dataset particularly in distinguishing the Setosa class is a contributing factor to the successful performance of Logistic Regression as a linear model. In a similar vein, the Decision Tree model is capable of generating unambiguous decision rules that precisely distinguish the classes according to defined feature thresholds.

The Random Forest model, on the other hand, obtained slightly lower scores across all metrics despite still demonstrating strong performance. The stochastic nature of ensemble methods may be responsible for this marginal reduction as the introduction of minor variability in predictions can be attributed to randomisation in sampling and feature selection. Such variability may have a more pronounced effect on performance due to the dataset's small size. In general, the performance of all three models is satisfactory. However, the Logistic Regression and Decision Tree is preferable on this dataset. Nevertheless, it is crucial to evaluate the most suitable model in terms of model complexity, generalisability, and accuracy.

#### Best Model Justification


---
The preferable model for this study is Logistic Regression, despite the fact that both Decision Tree and Logistic Regression achieved identical and perfect evaluation scores. The primary justification for this decision is its generalisability and simplicity. Logistic Regression is a parametric model that is less complex and has fewer assumptions than Decision Tree models. Decision Tree models are more susceptible to overfitting, particularly when trained on limited datasets. Although the Decision Tree model is an exact match for the training and testing data, it may capture noise and dataset-specific patterns that are not easily generalisable to unobserved data.

Additionally, Logistic Regression offers a framework that is more stable and interpretable, enabling a more comprehensive comprehension of the relationship between the predicted outcome and the input features. This is especially beneficial in real-world applications where model transparency and explainability are critical. Despite the fact that Random Forest is generally regarded as a more robust model due to its ensemble nature, its slightly lower performance in this instance combined with increased computational complexity renders it less suited than Logistic Regression for this particular dataset. Consequently, Logistic Regression is chosen as the most effective model due to its ability to achieve optimal performance while simultaneously preserving simpleness, interpretability and a greater potential for generalisation.




### Conclusion

This project illustrated the utilisation of Spark MLlib to execute a classification assignment on the Iris dataset. Cross-validation and grid search techniques were employed to optimise three models which are Logistic Regression, Decision Tree, and Random Forest.

The results indicate that both Logistic Regression and Decision Tree obtained perfect classification performance, while Random Forest exhibited slightly lower but still robust performance. As the most appropriate model, Logistic Regression was chosen due to its generalisation capability, interpretability, and simplicity.

Overall, this investigation underscores the necessity of meticulous model selection, tuning, and evaluation to guarantee reliable and resilient machine learning results even for datasets that are relatively straightforward.